# Prepare the data

Cuts the three scenes into tiles and burns my outlines into label rasters.

Each scene gets tiled in its own coordinate system. I resample to 0.125 m/px but
do not warp anything onto a shared grid, so the labels sit exactly where I drew
them.

Three files per tile:

```
<scene>_<col>_<row>.tif            image, uint16, raw values
<scene>_<col>_<row>_masks.tif      one integer per boulder
<scene>_<col>_<row>_corridor.tif   1 where I actually looked
index.json                         bounds, fold and counts per tile
```

Runs on the laptop, the source scenes are here. Only the finished archive goes
to the server.

In [ ]:
import sys
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt

sys.path.insert(0, ".")
import config
from common import tiles, folds

config.check_paths()
config.make_dirs()

print("raw    ", config.RAW)
print("archive", config.ARCHIVE)
print(f"{config.RES} m/px, tile {config.TILE} px = {config.TILE * config.RES:.0f} m, "
      f"overlap {config.OVERLAP} px = {config.OVERLAP * config.RES:.0f} m")

## What we are starting from

In [ ]:
rows = []
for s, info in config.SCENES.items():
    f = config.image(s)
    with rasterio.open(f) as r:
        rows.append(dict(scene=s, date=info["date"], file=f.name,
                         bands=r.count, dtype=str(r.dtypes[0]),
                         res=round(r.res[0], 4),
                         px=f"{r.width} x {r.height}",
                         upsample=round(r.res[0] / config.RES, 2)))

pd.DataFrame(rows).set_index("scene")

## Outlines

Quick check that these are the files I think they are. `_bloecke` is in each
scene's own coordinates and lines up with its image, `_bloecke_nov` is the same
rows moved into november's system, which is where the valley axis lives.

In [ ]:
outlines = {}
rows = []
for s in config.SCENES:
    g = gpd.read_file(config.OUTLINES / f"{s}_bloecke.gpkg").to_crs(config.CRS)
    g["d_m"] = 2 * np.sqrt(g.area / np.pi)
    outlines[s] = g

    nov = gpd.read_file(config.outlines(s))
    assert len(nov) == len(g), f"{s}: row counts differ, the two files are out of sync"

    ok = g.d_m >= config.MIN_DIAMETER
    rows.append(dict(scene=s, total=len(g),
                     training=int((g.d_m >= config.MIN_TRAINING).sum()),
                     scored=int(ok.sum()),
                     D50=round(g.d_m[ok].median(), 2),
                     D84=round(g.d_m[ok].quantile(0.84), 2),
                     largest=round(g.d_m.max(), 1)))

pd.DataFrame(rows).set_index("scene")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
for s, g in outlines.items():
    ax.hist(g.d_m[g.d_m < 15], bins=60, histtype="step", lw=1.4, label=s)
ax.axvline(config.MIN_DIAMETER, color="k", ls="--", lw=1)
ax.set_xlabel("equivalent diameter (m)")
ax.set_ylabel("count")
ax.set_title(f"dashed line is the {config.MIN_DIAMETER} m scoring limit")
ax.legend()
plt.tight_layout()
plt.show()

## Overlapping outlines

A label raster holds one integer per pixel, so two outlines that overlap cannot
both survive, the second one overwrites the first. Worth knowing how often that
happens before it quietly eats boulders.

In [ ]:
rows = []
for s, g in outlines.items():
    g = g[g.d_m >= config.MIN_TRAINING].reset_index(drop=True)
    pairs = gpd.sjoin(g[["geometry"]], g[["geometry"]], predicate="overlaps")
    n = len(pairs) // 2
    if n:
        a = g.geometry.values[pairs.index.values]
        b = g.geometry.values[pairs.index_right.values]
        frac = np.array([x.intersection(y).area / x.area for x, y in zip(a, b)])
        rows.append(dict(scene=s, pairs=n,
                         median_pct=round(float(np.median(frac)) * 100, 1),
                         max_pct=round(float(frac.max()) * 100, 1)))
    else:
        rows.append(dict(scene=s, pairs=0, median_pct=0.0, max_pct=0.0))

pd.DataFrame(rows).set_index("scene")

## Folds

Cuts are absolute kilometres along the valley, the same in every scene. The two
ends are always training because one scene is missing there: february starts at
km 7.7, march stops at 24.7.

In [ ]:
print("test block ", config.TEST_FROM, "-", config.TEST_TO, "km")
print("fold edges ", config.FOLD_EDGES)
print("always train", config.ALWAYS_TRAIN)

rows = {}
for s in config.SCENES:
    nov = gpd.read_file(config.outlines(s)).to_crs(config.CRS)
    d = 2 * np.sqrt(nov.area / np.pi)
    rows[s] = folds.summary(nov[d >= config.MIN_DIAMETER])

order = ["always_train", "fold1", "fold2", "fold3", "fold4", "test"]
df = pd.DataFrame(rows).T.reindex(columns=order).fillna(0).astype(int)
df.loc["all"] = df.sum()
df

## Build the archive

A few minutes per scene, each one reads and resamples around 150 Mpx.

In [ ]:
index = []
for s in config.SCENES:
    print(f"{s}:")
    index += tiles.build_scene(s)

path = tiles.write_index(index)
print(f"\n{len(index)} tiles -> {path}")

## What came out

In [ ]:
idx = pd.DataFrame(tiles.read_index())

per_scene = idx.groupby("scene").agg(
    tiles=("name", "size"),
    with_labels=("n_inner", lambda c: int((c > 0).sum())),
    boulders=("n_inner", "sum"),
    median_per_tile=("n_inner", "median"),
    corridor_share=("corridor", "mean"),
).round(2)
per_scene

In [ ]:
print(idx.groupby("fold").size().to_string())
empty = (idx.n_inner == 0).mean()
print(f"\nempty tiles: {empty:.0%}")
print(f"disk: {sum(f.stat().st_size for f in config.ARCHIVE.glob('*.tif')) / 1e9:.1f} GB")

## Look at a few tiles

Red is the label outline, blue the corridor edge. If these do not line up,
something upstream is wrong.

In [ ]:
pick = idx[idx.n_inner >= 8].head(6)

fig, axes = plt.subplots(2, 3, figsize=(13, 9))
for ax, (_, t) in zip(axes.ravel(), pick.iterrows()):
    with rasterio.open(config.ARCHIVE / f"{t['name']}.tif") as r:
        img = r.read(1).astype(float)
    with rasterio.open(config.ARCHIVE / f"{t['name']}_masks.tif") as r:
        lab = r.read(1)
    with rasterio.open(config.ARCHIVE / f"{t['name']}_corridor.tif") as r:
        corr = r.read(1)

    good = img > 0
    lo, hi = np.percentile(img[good], config.PERCENTILE)
    ax.imshow(np.clip((img - lo) / max(hi - lo, 1), 0, 1), cmap="gray")
    ax.contour(lab > 0, levels=[0.5], colors="red", linewidths=0.7)
    ax.contour(corr, levels=[0.5], colors="deepskyblue", linewidths=0.6)
    ax.set_title(f"{t['name']}  {t['n_inner']} boulders  [{t['fold']}]", fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

## Copy it over

```bash
rsync -av --progress \
  "$TRISHULI_DATA/archive/" \
  pcpool:/DATA/schrepel/trishuli_daten/archive/
```